# 27 · Annealing with and without slack variables: instance ladder benchmark

This notebook compares two QUBO encodings of GPU capacity, solved with local simulated annealing, against two Gurobi references across an instance ladder.

| Method | Formulation | Role |
|---|---|---|
| **SA-slack** | QUBO with binary GPU-slack variables | Annealing result for the exact slack equality encoding |
| **SA-unbalanced** | Slack-free QUBO with an unbalanced GPU penalty | Annealing result for a lower-variable-count surrogate |
| **Gurobi aligned MILP** | Same feasible scheduling choices and effective-price energy objective | Common energy-cost reference; assignment and GPU conditions are hard constraints |
| **Gurobi full MILP** | Physical renewable/grid dispatch, PUE and grid-import peak charge | Operational cost reference |

The QUBO-aligned MILP is not the same mathematical objective as every QUBO. For a feasible schedule the slack equality penalty is zero, while the unbalanced penalty can still vary across feasible GPU-load profiles. We therefore compare decoded schedules using the **effective-price energy cost** and separately report the physical cost. The reported energy-cost difference is descriptive; it is not an optimality gap for the full unbalanced BQM.

### GPU-capacity terms

For cluster $k$ and slot $t$, let $L_{k,t}(x)$ be active GPU demand, $G_k$ capacity and $h_{k,t}=G_k-L_{k,t}$ free margin:

$$H^{\text{slack}}_G=\lambda_G\sum_{k,t}\Big(L_{k,t}+\sum_b2^b s_{k,t,b}-G_k\Big)^2,\qquad
H^{\text{UP}}_G=\sum_{k,t}\big[-\lambda_1h_{k,t}+\lambda_2h_{k,t}^2\big].$$

### Comparison protocol

- The **primary comparison turns the QUBO peak proxy off for both encodings**, so the GPU encoding is the only formulation difference.
- A separate paired peak-proxy ablation uses the same selected assignment/GPU weights for both encodings on representative levels.
- Penalty weights are tuned per instance on a tuning seed and evaluated on three fresh annealing seeds. This is a per-instance best-case study, not a test of generalizable penalty weights.
- Feasibility is checked after decoding. Candidate schedules are selected by feasibility first and effective-price energy cost second; this classical post-selection is part of the reported workflow.
- Simulated annealing is a classical heuristic, not quantum hardware evidence or a quantum-advantage claim.

## 1. Setup and run settings

Penalty grids use dimensionless multipliers of each instance's energy scale $E$, defined as the highest effective-price cost among its feasible single-job assignment options. The code records the corresponding absolute penalty values. The two GPU encodings have different weight parameters, but both use the same assignment grid and **peak multiplier $p=0$ in the primary comparison**.

The selected weights are tuned separately for each instance with `TUNING_SEED`; the reported annealing results use three fresh seeds. Because tuning and evaluation use the same instance, interpret the result as instance-specific calibration rather than out-of-sample generalization.

In [ ]:
from __future__ import annotations

import itertools
import json
import math
import sys
import time
from pathlib import Path

import gurobipy as gp
from gurobipy import GRB
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

REPO_ROOT = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / 'src').is_dir() and (folder / 'pyproject.toml').exists()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError('Start the kernel inside the data_center_scheduling repository.')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from dwave.samplers import SimulatedAnnealingSampler

from src.config import ModelConfig
from src.milp.gurobi_model import build_milp_model
from src.milp.solve import solve_model
from src.quantum.benchmarking import (
    add_effective_price_for_qubo,
    evaluate_physical_schedule,
    make_effective_price_qubo_inputs,
)
from src.quantum.qubo_builder import (
    QuboPenaltyWeights,
    UnbalancedGpuPenaltyWeights,
    build_scheduling_qubo,
    decode_qubo_sample,
    summarize_qubo_graph,
    to_dimod_bqm,
    validate_decoded_schedule,
)

In [ ]:
# Levels to run (see section 2). Remove entries to shorten a run.
RUN_LEVELS = ('L0', 'L1', 'L2', 'L3', 'L4', 'L5', 'L6', 'L7')

# Annealing.
TUNING_READS = 100          # reads per penalty configuration during the search
FINAL_READS = 1_000         # reads for the reported run with the selected weights
TUNING_SEED = 11
FINAL_SEEDS = (2026, 2027, 2028)  # independent final annealing seeds
NUM_SWEEPS = 1_000          # D-Wave SimulatedAnnealingSampler default

# Penalty search grids: multipliers of the instance energy scale E.
ASSIGNMENT_MULTIPLIERS = (2.0, 10.0)                 # lambda_A = a * E
SLACK_GPU_MULTIPLIERS = (0.01, 0.1, 1.0, 10.0)       # lambda_G = g * E
SLACK_PEAK_MULTIPLIERS = (0.0,)                      # peak proxy off in the primary comparison
UP_QUADRATIC_MULTIPLIERS = (0.01, 0.1, 1.0, 10.0)    # lambda_2 = q * E
UP_RESERVE_FRACTIONS = (0.10, 0.25)                  # h* = r * capacity, lambda_1 = 2 * lambda_2 * h*
UP_PEAK_MULTIPLIERS = (0.0,)                         # peak proxy off in the primary comparison

# Absolute weights that bypass the search, e.g.
# MANUAL_WEIGHTS[('L7', 'slack')] = {'assignment': 2e4, 'gpu': 50.0, 'peak': 0.0}
# MANUAL_WEIGHTS[('L7', 'unbalanced')] = {'assignment': 2e4, 'up_linear': 300.0, 'up_quadratic': 40.0, 'peak': 0.0}
MANUAL_WEIGHTS: dict[tuple[str, str], dict[str, float]] = {}

# Gurobi.
MILP_TIME_LIMIT = 60
MILP_GAP = 1e-4
THREADS = 8
RUN_FULL_MILP = True

# Physical evaluation, identical to the full MILP and to notebooks 21/26.
CONTRACTED_POWER_RATIO = 0.70
PEAK_PRICE = 1_000.0

# Use a protocol-specific output directory so older peak-on tuning files cannot be reused.
REUSE_SAVED_TUNING = True  # only reuse files produced under this versioned protocol
RESULTS_DIR = REPO_ROOT / 'results/annealing_ladder_peakoff_v2'
FIGURES_DIR = RESULTS_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

ENCODINGS = ('slack', 'unbalanced')
METHOD_LABELS = {
    'slack': 'SA-slack', 'unbalanced': 'SA-unbalanced',
    'milp_aligned': 'Gurobi QUBO-aligned MILP', 'milp_full': 'Gurobi full MILP',
}
print(f'Gurobi {".".join(map(str, gp.gurobi.version()))}; threads={THREADS}; results -> {RESULTS_DIR.relative_to(REPO_ROOT)}')

## 2. Instance ladder

| Level | Instance | Slots | Purpose |
|---|---|---|---|
| L0 | Four-job toy instance from notebook 13_2 | 3 × 1 h | Hand-checks assignment, GPU capacity and decoding |
| L1 | Seren tiny, 6 jobs | 24 × 1 h | Small trace-derived case |
| L2 | Seren medium, 24 jobs | 24 × 1 h | Larger trace-derived case with loose capacity |
| L3 | Seren max-day, 65 jobs | 24 × 1 h | Largest trace-derived hourly case |
| L4 | Seren medium, 24 jobs, tight cluster | 24 × 1 h | Controlled capacity-binding variant |
| L5 | Seren max-day, 65 jobs, tight cluster | 24 × 1 h | Higher-contention hourly variant |
| L6 | Synthetic A100, 42 jobs, `long_base` seed 7601 | 96 × 15 min | 15-minute multi-node instance |
| L7 | Synthetic A100, 42 jobs, `long_wide_window` seed 7602 | 96 × 15 min | Wider-window target benchmark |

This is an **instance ladder**, not a controlled comparison of 1-hour versus 15-minute resolution: workload source, capacity pressure and temporal resolution vary together. The tight Seren cases change cluster capacity to create controlled stress cases; report them as synthetic capacity variants, not as observed cluster configurations.

**Lossless unit normalisation.** GPU demands and capacities are divided by their greatest common divisor (16 nodes on L6–L7, 1 on Seren). Multiplying back reproduces the physical demands. Report both the normalized solver unit and its physical interpretation.

In [ ]:
SEREN_ROOT = REPO_ROOT / 'data/instances/seren_pretrain'
STRESS_ROOT = REPO_ROOT / 'data/instances/single_cluster_15m/long_duration_focused'

LEVEL_SPECS = {
    'L0': {'name': 'toy_4_jobs_nb13_2', 'source': 'toy'},
    'L1': {'name': 'seren_tiny_6', 'source': 'seren', 'folder': 'tiny_6_jobs'},
    'L2': {'name': 'seren_medium_24', 'source': 'seren', 'folder': 'medium_24_jobs'},
    'L3': {'name': 'seren_max_day_65', 'source': 'seren', 'folder': 'max_day_jobs'},
    'L4': {'name': 'seren_medium_24_tight', 'source': 'seren', 'folder': 'medium_24_jobs', 'target_utilization': 0.60},
    'L5': {'name': 'seren_max_day_65_tight', 'source': 'seren', 'folder': 'max_day_jobs', 'target_utilization': 0.70},
    'L6': {'name': 'a100_15min_long_base_7601', 'source': 'stress', 'folder': 'long_base_seed_7601'},
    'L7': {'name': 'a100_15min_wide_window_7602', 'source': 'stress', 'folder': 'long_wide_window_seed_7602'},
}


def toy_instance():
    # Notebook 13_2 walkthrough instance: 4 jobs, one cluster with 2 GPU units, 3 hourly slots.
    # j1 lasts 2 slots (starts 0-1); j2-j4 last 1 slot (starts 0-2). Every job needs
    # 1 GPU unit. Power, duration and windows match notebook 13_2. Effective prices
    # are 30/36/42 EUR/MWh, fixed PUE is 1.20, and renewables are unavailable.
    def job(job_id, duration, latest_start, power):
        return {'job_id': job_id, 'category': 'gpu', 'duration': duration, 'power': power,
                'earliest_start': 0, 'latest_start': latest_start, 'gpu_count_required': 1,
                'compatible_clusters': 'c_main'}

    jobs = pd.DataFrame([
        job('j1', 2, 1, 0.030), job('j2', 1, 2, 0.025),
        job('j3', 1, 2, 0.015), job('j4', 1, 2, 0.005),
    ])
    # Capacity matches two GPU units; power capacity is non-binding for any GPU-feasible slot.
    clusters = pd.DataFrame([{'cluster_id': 'c_main', 'capacity': 0.055, 'gpu_count': 2, 'gpu_capacity': 2}])
    price = [30.0, 36.0, 42.0]
    energy = pd.DataFrame({
        'hour': [0, 1, 2], 'grid_price': price, 'effective_price': price,
        'renewable_available': [0.0, 0.0, 0.0], 'renewable_price': [24.0, 24.0, 24.0],
        'pue': [1.20, 1.20, 1.20], 'baseline_load': [0.0, 0.0, 0.0],
    })
    return jobs, clusters, energy, 1.0


def seren_instance(folder: str):
    directory = SEREN_ROOT / folder
    energy = pd.read_csv(directory / 'hourly_inputs.csv')
    if 'pue' not in energy:
        energy['pue'] = 1.25
    if 'baseline_load' not in energy:
        energy['baseline_load'] = 0.0
    return pd.read_csv(directory / 'jobs.csv'), pd.read_csv(directory / 'clusters.csv'), energy, 1.0


def stress_instance(folder: str):
    directory = STRESS_ROOT / folder
    return (pd.read_csv(directory / 'jobs.csv'), pd.read_csv(directory / 'clusters.csv'),
            pd.read_csv(directory / 'energy_inputs_15m.csv'), 0.25)


def normalize_gpu_units(jobs: pd.DataFrame, clusters: pd.DataFrame):
    demands = jobs['gpu_count_required'].astype(int).to_numpy()
    capacities = clusters['gpu_count'].astype(int).to_numpy()
    unit = int(np.gcd.reduce(np.concatenate([demands, capacities])))
    jobs, clusters = jobs.copy(), clusters.copy()
    jobs['gpu_count_required'] = jobs['gpu_count_required'].astype(int) // unit
    clusters['gpu_count'] = clusters['gpu_count'].astype(int) // unit
    clusters['gpu_capacity'] = clusters['gpu_count']
    if not (jobs['gpu_count_required'] * unit).equals(pd.Series(demands, index=jobs.index)):
        raise AssertionError('GPU normalisation is not lossless')
    return jobs, clusters, unit


def aligned_config(clusters: pd.DataFrame, energy: pd.DataFrame, delta_t: float) -> ModelConfig:
    # The QUBO-aligned configuration used by notebooks 21, 22 and 26.
    return ModelConfig(
        contracted_power=float(clusters['capacity'].sum()) * 10.0, renewable_price=1_000_000.0,
        peak_price=0.0, delta_t=delta_t, pue=float(energy['pue'].iloc[0]),
    )


def aligned_milp(jobs, clusters, energy, delta_t):
    return build_milp_model(
        jobs, make_effective_price_qubo_inputs(energy), clusters, aligned_config(clusters, energy, delta_t),
        model_name='qubo_aligned_milp',
        enforce_gpu_constraints=True, enforce_cpu_constraints=False, enforce_memory_constraints=False,
    )


def is_feasible(jobs, clusters, energy, delta_t) -> bool:
    model, _ = aligned_milp(jobs, clusters, add_effective_price_for_qubo(energy, jobs, clusters), delta_t)
    model.Params.OutputFlag = 0
    model.Params.Threads = THREADS
    try:
        solve_model(model, time_limit=30, raise_on_no_solution=False)
    except RuntimeError:
        return False
    return model.SolCount > 0


def tighten_cluster(jobs, clusters, energy, delta_t, target_utilization: float):
    # Shrink one cluster to the target utilisation, then grow until feasible.
    if len(clusters) != 1:
        raise ValueError('capacity tightening is defined for single-cluster instances')
    demand = float((jobs['gpu_count_required'] * jobs['duration']).sum())
    original = int(clusters['gpu_count'].iloc[0])
    capacity = max(int(jobs['gpu_count_required'].max()), math.ceil(demand / (len(energy) * target_utilization)))
    while capacity <= original:
        scale = capacity / original
        tight = clusters.copy()
        tight['gpu_count'] = capacity
        tight['gpu_capacity'] = capacity
        for column in ('capacity', 'capacity_kw', 'power_capacity_kw'):
            if column in tight:
                tight[column] = tight[column] * scale
        if is_feasible(jobs, tight, energy, delta_t):
            return tight
        capacity += 1
    return clusters


def load_level(level: str) -> dict:
    spec = LEVEL_SPECS[level]
    if spec['source'] == 'toy':
        jobs, clusters, energy, delta_t = toy_instance()
    elif spec['source'] == 'seren':
        jobs, clusters, energy, delta_t = seren_instance(spec['folder'])
    else:
        jobs, clusters, energy, delta_t = stress_instance(spec['folder'])
    jobs, clusters, unit = normalize_gpu_units(jobs, clusters)
    if 'target_utilization' in spec:
        clusters = tighten_cluster(jobs, clusters, energy, delta_t, spec['target_utilization'])
    if 'effective_price' not in energy:
        energy = add_effective_price_for_qubo(energy, jobs, clusters)
    if 'renewable_price' not in energy:
        energy['renewable_price'] = 0.8 * float(energy['grid_price'].min())
    return {'level': level, **spec, 'jobs': jobs, 'clusters': clusters, 'energy': energy,
            'delta_t': delta_t, 'gpu_unit': unit, 'hours': energy['hour'].astype(int).tolist()}


def option_costs(instance: dict) -> list[np.ndarray]:
    # Effective-price energy cost of every start option: the QUBO energy coefficient.
    energy = instance['energy'].sort_values('hour')
    weighted = np.concatenate([[0.0], np.cumsum(energy['effective_price'].to_numpy(float) * energy['pue'].to_numpy(float))])
    costs = []
    for job in instance['jobs'].itertuples(index=False):
        starts = np.arange(int(job.earliest_start), int(job.latest_start) + 1)
        costs.append(float(job.power) * instance['delta_t'] * (weighted[starts + int(job.duration)] - weighted[starts]))
    return costs


instances = {}
for level in RUN_LEVELS:
    instance = load_level(level)
    instance['energy_scale'] = max(float(costs.max()) for costs in option_costs(instance))
    instance['peak_reference_mw'] = float(instance['clusters']['capacity'].sum()) * float(instance['energy']['pue'].iloc[0])
    instances[level] = instance

instance_table = pd.DataFrame([
    {
        'level': level, 'instance': data['name'], 'jobs': len(data['jobs']), 'slots': len(data['energy']),
        'slot_minutes': int(round(60 * data['delta_t'])), 'clusters': len(data['clusters']),
        'gpu_unit': data['gpu_unit'], 'capacity_units': int(data['clusters']['gpu_count'].sum()),
        'utilization': float((data['jobs']['gpu_count_required'] * data['jobs']['duration']).sum()
                             / (data['clusters']['gpu_count'].sum() * len(data['energy']))),
        'mean_duration_slots': float(data['jobs']['duration'].mean()),
        'mean_start_options': float((data['jobs']['latest_start'] - data['jobs']['earliest_start'] + 1).mean()),
        'energy_scale_E': data['energy_scale'],
    }
    for level, data in instances.items()
])
instance_table.to_csv(RESULTS_DIR / 'instances.csv', index=False)
display(instance_table)

## 3. QUBO formulations

`build_qubo` constructs the two GPU-capacity encodings using absolute weights. Both use the same assignment variables, effective-price energy term, one-hot assignment penalty and (in the primary comparison) zero peak-proxy weight.

- **Slack:** $λ_G$ penalizes the equality between active GPU demand plus binary unused-capacity bits and cluster capacity. Every feasible GPU load can be represented with zero capacity penalty.
- **Unbalanced, slack-free:** $λ_1$ and $λ_2$ shape a quadratic function of free GPU margin. It removes slack bits but can assign different penalty values to distinct feasible load profiles.

For each instance, $E$ scales the penalty grid. The unbalanced formulation sets $λ_2=qE$ and $λ_1=2λ_2h^*$, where $h^*=rG$ is the preferred free margin. In the primary experiment $p=0$ for both encodings. The separate ablation varies $p$ while holding the selected assignment/GPU weights fixed.

In [ ]:
def build_qubo(instance: dict, encoding: str, weights: dict):
    penalty = QuboPenaltyWeights(
        assignment=weights['assignment'],
        gpu_capacity=weights.get('gpu', 0.0) if encoding == 'slack' else 0.0,
        peak_smoothing=weights.get('peak', 0.0),
    )
    unbalanced = (
        UnbalancedGpuPenaltyWeights(linear=weights['up_linear'], quadratic=weights['up_quadratic'])
        if encoding == 'unbalanced' else None
    )
    return build_scheduling_qubo(
        instance['jobs'], make_effective_price_qubo_inputs(instance['energy']), instance['clusters'],
        aligned_config(instance['clusters'], instance['energy'], instance['delta_t']), penalty,
        unbalanced_gpu_weights=unbalanced,
    )


def weights_from_multipliers(instance: dict, encoding: str, a: float, gpu_or_q: float,
                             reserve: float = 0.0, p: float = 0.0) -> dict:
    scale = instance['energy_scale']
    weights = {'assignment': a * scale, 'peak': p * scale / instance['peak_reference_mw'] ** 2}
    if encoding == 'slack':
        weights['gpu'] = gpu_or_q * scale
    else:
        capacity = float(instance['clusters']['gpu_count'].max())
        weights['up_quadratic'] = gpu_or_q * scale
        weights['up_linear'] = 2.0 * weights['up_quadratic'] * reserve * capacity
    return weights


def qubo_structure(qubo) -> dict:
    coefficients = np.abs(np.array([*qubo.linear.values(), *qubo.quadratic.values()], dtype=float))
    coefficients = coefficients[coefficients > 0]
    return {
        **summarize_qubo_graph(qubo),
        'max_abs_coefficient': float(coefficients.max()) if coefficients.size else 0.0,
        'coefficient_dynamic_range': float(coefficients.max() / coefficients.min()) if coefficients.size else 0.0,
    }

## 4. Annealing, decoding and schedule metrics

Each QUBO is sampled, decoded and checked against the original assignment, start-window and GPU-capacity constraints. Within a run, the selection rule is lexicographic: feasible schedules first, then the lowest effective-price energy cost; infeasible samples are ranked by assignment errors, overloaded slots and overload size.

This means the reported schedule is the best feasible candidate found in the sample pool after classical validation and post-selection. It is not necessarily the raw lowest-QUBO-energy sample. The reported effective-price cost is kept separate from QUBO energy and from the physically re-evaluated cost.

Only the sampler call is timed for annealing; decoding and validation are excluded. Runtime is implementation-specific and must not be interpreted as a hardware speed comparison between local simulated annealing and Gurobi.

In [ ]:
SAMPLER = SimulatedAnnealingSampler()


def schedule_energy_cost(schedule: pd.DataFrame, instance: dict) -> float:
    # Effective-price energy component matches the aligned MILP objective; this is not total BQM energy.
    if schedule.empty:
        return float('nan')
    energy = instance['energy'].set_index('hour')
    unit_price = energy['effective_price'] * energy['pue']
    total = 0.0
    for row in schedule.itertuples(index=False):
        slots = range(int(row.start_hour), int(row.start_hour) + int(row.duration))
        total += float(row.power) * instance['delta_t'] * float(unit_price.loc[list(slots)].sum())
    return total


def violation_summary(schedule: pd.DataFrame, instance: dict) -> dict:
    jobs, clusters = instance['jobs'], instance['clusters']
    counts = schedule['job_id'].value_counts() if not schedule.empty else pd.Series(dtype=int)
    missing = sum(int(counts.get(job_id, 0)) == 0 for job_id in jobs['job_id'])
    extras = sum(max(0, int(count) - 1) for count in counts)
    overloaded_slots, max_excess = 0, 0.0
    for cluster in clusters.itertuples(index=False):
        selected = schedule[schedule['assigned_cluster'] == cluster.cluster_id] if not schedule.empty else schedule
        for hour in instance['hours']:
            if selected.empty:
                continue
            active = selected[(selected['start_hour'] <= hour) & (hour < selected['start_hour'] + selected['duration'])]
            excess = float(active['gpu_count_required'].sum()) - float(cluster.gpu_count)
            if excess > 1e-9:
                overloaded_slots += 1
                max_excess = max(max_excess, excess)
    return {'assignment_errors': missing + extras, 'overloaded_slots': overloaded_slots,
            'max_overload_gpu': max_excess * instance['gpu_unit']}


def anneal(instance: dict, encoding: str, weights: dict, *, num_reads: int, seed: int) -> dict:
    qubo = build_qubo(instance, encoding, weights)
    started = time.perf_counter()
    sampleset = SAMPLER.sample(to_dimod_bqm(qubo), num_reads=num_reads, num_sweeps=NUM_SWEEPS, seed=seed)
    sampling_time = time.perf_counter() - started

    feasible_reads, candidates = 0, []
    for record in sampleset.aggregate().data(fields=['sample', 'energy', 'num_occurrences']):
        schedule = decode_qubo_sample(qubo, dict(record.sample))
        feasible = validate_decoded_schedule(schedule, instance['jobs'], instance['clusters'], instance['hours'])['feasible']
        feasible_reads += int(record.num_occurrences) if feasible else 0
        candidates.append({'schedule': schedule, 'feasible': feasible, 'qubo_energy': float(record.energy),
                           'energy_cost': schedule_energy_cost(schedule, instance),
                           **violation_summary(schedule, instance)})
    best = min(candidates, key=lambda c: (
        not c['feasible'], c['assignment_errors'], c['overloaded_slots'], c['max_overload_gpu'],
        c['energy_cost'] if c['feasible'] else c['qubo_energy'],
    ))
    return {
        'encoding': encoding, 'num_reads': num_reads, 'sampling_time_s': sampling_time,
        'feasible_fraction': feasible_reads / num_reads, 'feasible': best['feasible'],
        'energy_cost': best['energy_cost'] if best['feasible'] else float('nan'),
        'qubo_energy': best['qubo_energy'], 'assignment_errors': best['assignment_errors'],
        'overloaded_slots': best['overloaded_slots'], 'max_overload_gpu': best['max_overload_gpu'],
        'schedule': best['schedule'], **{f'w_{key}': value for key, value in weights.items()},
    }

## 5. Gurobi reference solves

Both MILPs are built with the repository's model and solved with a 60-second time limit and a target relative MIP gap of $10^{-4}$. The **aligned MILP** minimizes the effective-price energy cost with assignment and GPU capacity enforced as hard constraints. The **full MILP** uses physical renewable/grid dispatch, fixed PUE and grid-import peak charges (contracted power at 70% of facility capacity; peak price 1,000).

The aligned MILP is a common energy-cost benchmark, not a universal optimum for each BQM: the slack-free penalty may influence its feasible schedules. We report Gurobi's incumbent, MIP gap and whether optimality was proven. If the time or gap limit is reached before Gurobi proves optimality, call it an incumbent/reference solution rather than an optimum.

In [ ]:
ENERGY_COLUMNS = ['hour', 'renewable_available', 'grid_price', 'pue', 'baseline_load']


def full_config(instance: dict) -> ModelConfig:
    clusters, energy = instance['clusters'], instance['energy']
    pue = float(energy['pue'].iloc[0])
    return ModelConfig(
        contracted_power=float(clusters['capacity'].sum()) * pue * CONTRACTED_POWER_RATIO,
        renewable_price=float(energy['renewable_price'].iloc[0]), peak_price=PEAK_PRICE,
        delta_t=instance['delta_t'], pue=pue,
    )


def extract_milp_schedule(variables: dict, jobs: pd.DataFrame) -> pd.DataFrame:
    rows, jobs_by_id = [], jobs.set_index('job_id')
    for (job_id, cluster, start), variable in variables['x'].items():
        if variable.X > 0.5:
            job = jobs_by_id.loc[job_id]
            rows.append({'job_id': job_id, 'assigned_cluster': cluster, 'start_hour': int(start),
                         'duration': int(job.duration), 'power': float(job.power),
                         'gpu_count_required': float(job.gpu_count_required)})
    return pd.DataFrame(rows)


def solve_milp(instance: dict, kind: str) -> dict:
    jobs, clusters, energy = instance['jobs'], instance['clusters'], instance['energy']
    if kind == 'milp_aligned':
        model, variables = aligned_milp(jobs, clusters, energy, instance['delta_t'])
    else:
        model, variables = build_milp_model(
            jobs, energy[ENERGY_COLUMNS], clusters, full_config(instance), model_name='full_milp',
            enforce_gpu_constraints=True, enforce_cpu_constraints=False, enforce_memory_constraints=False,
        )
    model.Params.OutputFlag = 0
    model.Params.Threads = THREADS
    started = time.perf_counter()
    solve_model(model, time_limit=MILP_TIME_LIMIT, mip_gap=MILP_GAP, raise_on_no_solution=False)
    runtime = time.perf_counter() - started
    has_solution = model.SolCount > 0
    schedule = extract_milp_schedule(variables, jobs) if has_solution else pd.DataFrame()
    return {
        'encoding': kind, 'sampling_time_s': runtime, 'feasible': has_solution,
        'feasible_fraction': float(has_solution), 'status': int(model.Status),
        'mip_gap': float(model.MIPGap) if has_solution else None,
        'mip_status': int(model.Status),
        'optimality_proven': bool(model.Status == GRB.OPTIMAL),
        'objective': float(model.ObjVal) if has_solution else None,
        'energy_cost': schedule_energy_cost(schedule, instance) if has_solution else float('nan'),
        'schedule': schedule,
    }

## 6. Penalty search

For each level and GPU encoding, the penalty grid is sampled with `TUNING_READS` reads and `TUNING_SEED`. The peak proxy is fixed at zero for both encodings. Selection is lexicographic:

1. configurations with at least one feasible decoded sample;
2. among those, the lowest effective-price energy cost of the best feasible schedule;
3. ties broken by the larger feasible fraction.

If no configuration is feasible, retain the one closest to feasibility (fewest assignment errors, then overloaded slots, then overload size). QUBO energies are not compared across weight configurations because their scales differ. Tuning results are saved under the versioned `results/annealing_ladder_peakoff_v2/` directory.

In [ ]:
def search_grid(instance: dict, encoding: str) -> list[dict]:
    if encoding == 'slack':
        return [weights_from_multipliers(instance, 'slack', a, g, p=p) | {'grid': f'a={a}, g={g}, p={p}'}
                for a, g, p in itertools.product(ASSIGNMENT_MULTIPLIERS, SLACK_GPU_MULTIPLIERS, SLACK_PEAK_MULTIPLIERS)]
    return [weights_from_multipliers(instance, 'unbalanced', a, q, r, p) | {'grid': f'a={a}, q={q}, r={r}, p={p}'}
            for a, q, r, p in itertools.product(ASSIGNMENT_MULTIPLIERS, UP_QUADRATIC_MULTIPLIERS,
                                               UP_RESERVE_FRACTIONS, UP_PEAK_MULTIPLIERS)]


def selection_key(row) -> tuple:
    if row['feasible']:
        return (0, row['energy_cost'], -row['feasible_fraction'])
    return (1, row['assignment_errors'], row['overloaded_slots'], row['max_overload_gpu'])


def weights_of(row) -> dict:
    return {key[2:]: float(row[key]) for key in row.index if key.startswith('w_') and pd.notna(row[key])}


def tune(level: str, encoding: str) -> tuple[dict, pd.DataFrame, float]:
    path = RESULTS_DIR / f'tuning_{level}_{encoding}.csv'
    if (level, encoding) in MANUAL_WEIGHTS:
        return dict(MANUAL_WEIGHTS[(level, encoding)]), pd.DataFrame(), 0.0
    if REUSE_SAVED_TUNING and path.exists():
        table = pd.read_csv(path)
        return weights_of(table.loc[table['selected']].iloc[0]), table, float(table['sampling_time_s'].sum())
    rows = []
    for config in search_grid(instances[level], encoding):
        label = config.pop('grid')
        result = anneal(instances[level], encoding, config, num_reads=TUNING_READS, seed=TUNING_SEED)
        result.pop('schedule')
        rows.append({'level': level, 'grid': label, **result})
    table = pd.DataFrame(rows)
    best_index = min(table.index, key=lambda index: selection_key(table.loc[index]))
    table['selected'] = table.index == best_index
    table.to_csv(path, index=False)
    return weights_of(table.loc[best_index]), table, float(table['sampling_time_s'].sum())


selected_weights, tuning_tables, tuning_time = {}, {}, {}
for level in RUN_LEVELS:
    for encoding in ENCODINGS:
        started = time.perf_counter()
        weights, table, sampling = tune(level, encoding)
        selected_weights[(level, encoding)] = weights
        tuning_tables[(level, encoding)] = table
        tuning_time[(level, encoding)] = sampling
        feasible = int(table['feasible'].sum()) if not table.empty else 'manual'
        print(f'{level} {encoding:10s}: {len(table):2d} configs, {feasible} feasible, '
              f'{time.perf_counter() - started:6.1f}s; selected {weights}', flush=True)

In [ ]:
# Tuning overview: feasible configurations and selected point per level and formulation.
overview = []
for (level, encoding), table in tuning_tables.items():
    if table.empty:
        continue
    chosen = table.loc[table['selected']].iloc[0]
    overview.append({
        'level': level, 'encoding': encoding, 'configs': len(table),
        'feasible_configs': int(table['feasible'].sum()),
        'best_feasible_fraction': float(table['feasible_fraction'].max()),
        'selected_grid': chosen['grid'], 'selected_feasible': bool(chosen['feasible']),
        'selected_energy_cost': chosen['energy_cost'], 'tuning_sampling_time_s': tuning_time[(level, encoding)],
    })
tuning_overview = pd.DataFrame(overview)
display(tuning_overview)

### Paired peak-proxy ablation

The main comparison keeps peak smoothing off for both GPU encodings. The ablation later in the notebook tests peak weights $p\in\{0,0.1,1\}$ on L0, L4 and L7. For each encoding and level, assignment and GPU weights remain fixed at the values selected in the peak-off search; only the peak weight changes. This isolates the peak term and avoids selecting different peak settings during the primary GPU-encoding comparison.

These three levels are representative cases, not a controlled comparison of temporal resolution.

The paired ablation is run after the common primary weights have been selected, so it does not contaminate the primary formulation comparison.

## 7. Final runs

With the peak-off weights selected above, run each annealing formulation on three fresh seeds, each with `FINAL_READS`. Solve each MILP once per level. Save per-seed records, then summarize annealing feasibility fractions and cost metrics across seeds. For cost gaps, use the median feasible-run cost relative to the aligned/full MILP reference incumbent; do not label this a QUBO optimality gap.

In [ ]:
def physical_metrics(schedule: pd.DataFrame, instance: dict) -> dict:
    if schedule.empty:
        return {}
    contracted = CONTRACTED_POWER_RATIO * float(instance['clusters']['capacity'].sum()) * float(instance['energy']['pue'].iloc[0])
    _, summary = evaluate_physical_schedule(schedule, instance['energy'], delta_t=instance['delta_t'],
                                            contracted_power_mw=contracted, peak_price=PEAK_PRICE)
    return summary


final_run_rows = []
for level in RUN_LEVELS:
    instance = instances[level]
    for encoding in ENCODINGS:
        for seed in FINAL_SEEDS:
            run = anneal(instance, encoding, selected_weights[(level, encoding)],
                         num_reads=FINAL_READS, seed=seed)
            schedule = run.pop('schedule')
            physical = physical_metrics(schedule, instance) if run['feasible'] else {}
            final_run_rows.append({
                'level': level, 'instance': instance['name'], 'encoding': encoding,
                'method': METHOD_LABELS[encoding], 'replicate_seed': seed,
                **run, 'tuning_time_s': tuning_time.get((level, encoding), 0.0), **physical,
            })

    for encoding in ('milp_aligned', 'milp_full') if RUN_FULL_MILP else ('milp_aligned',):
        run = solve_milp(instance, encoding)
        schedule = run.pop('schedule')
        physical = physical_metrics(schedule, instance) if run['feasible'] else {}
        final_run_rows.append({
            'level': level, 'instance': instance['name'], 'method': METHOD_LABELS[encoding],
            'replicate_seed': None, **run, 'tuning_time_s': 0.0, **physical,
        })
    print(f'{level} done', flush=True)

final_runs = pd.DataFrame(final_run_rows)
final_runs.to_csv(RESULTS_DIR / 'final_runs_by_seed.csv', index=False)

# One row per method and level for thesis tables. SA costs are medians across feasible
# independent runs; feasible_fraction and runtime are means across the three seeds.
summary_rows = []
for (level, encoding), group in final_runs.groupby(['level', 'encoding'], sort=False):
    annealing_method = encoding in ENCODINGS
    feasible_group = group[group['feasible']]
    row = {
        'level': level, 'instance': group['instance'].iloc[0], 'encoding': encoding,
        'method': group['method'].iloc[0], 'feasible': bool(group['feasible'].any()),
        'feasible_fraction': float(group['feasible_fraction'].mean()),
        'feasible_fraction_sd': float(group['feasible_fraction'].std(ddof=1)) if annealing_method else 0.0,
        'feasible_replicates': int(group['feasible'].sum()),
        'n_replicates': int(group['replicate_seed'].notna().sum()) if annealing_method else 1,
        'sampling_time_s': float(group['sampling_time_s'].mean()),
        'tuning_time_s': float(group['tuning_time_s'].iloc[0]),
        'energy_cost': float(feasible_group['energy_cost'].median()) if not feasible_group.empty else float('nan'),
        'physical_total_cost': float(feasible_group['physical_total_cost'].median()) if not feasible_group.empty and 'physical_total_cost' in feasible_group else float('nan'),
        'facility_energy_mwh': float(feasible_group['facility_energy_mwh'].median()) if not feasible_group.empty and 'facility_energy_mwh' in feasible_group else float('nan'),
        'grid_energy_mwh': float(feasible_group['grid_energy_mwh'].median()) if not feasible_group.empty and 'grid_energy_mwh' in feasible_group else float('nan'),
        'renewable_energy_mwh': float(feasible_group['renewable_energy_mwh'].median()) if not feasible_group.empty and 'renewable_energy_mwh' in feasible_group else float('nan'),
        'physical_peak_grid_import_mw': float(feasible_group['physical_peak_grid_import_mw'].median()) if not feasible_group.empty and 'physical_peak_grid_import_mw' in feasible_group else float('nan'),
        'assignment_errors': float(group['assignment_errors'].median()) if 'assignment_errors' in group else float('nan'),
        'overloaded_slots': float(group['overloaded_slots'].median()) if 'overloaded_slots' in group else float('nan'),
        'max_overload_gpu': float(group['max_overload_gpu'].median()) if 'max_overload_gpu' in group else float('nan'),
        'mip_gap': group['mip_gap'].iloc[0] if 'mip_gap' in group else float('nan'),
        'optimality_proven': bool(group['optimality_proven'].iloc[0]) if 'optimality_proven' in group else None,
        'mip_status': group['mip_status'].iloc[0] if 'mip_status' in group else None,
    }
    summary_rows.append(row)

final = pd.DataFrame(summary_rows)
aligned_reference = final[final['encoding'] == 'milp_aligned'].set_index('level')['energy_cost']
final['energy_cost_gap_to_aligned_reference_pct'] = 100 * (final['energy_cost'] / final['level'].map(aligned_reference) - 1)
if RUN_FULL_MILP:
    physical_reference = final[final['encoding'] == 'milp_full'].set_index('level')['physical_total_cost']
    final['physical_cost_gap_to_full_milp_incumbent_pct'] = 100 * (final['physical_total_cost'] / final['level'].map(physical_reference) - 1)
final.to_csv(RESULTS_DIR / 'final_results_summary.csv', index=False)
display(final.head())

### 7.1 Paired peak-proxy ablation

This secondary experiment uses L0 (toy), L4 (capacity-tight hourly) and L7 (15-minute target). For each encoding, it holds the peak-off assignment/GPU weights fixed and varies only the peak multiplier. It reports feasibility, effective-price energy cost and physical peak/cost. The ablation is descriptive and does not alter the primary peak-off comparison or penalty selection.

In [ ]:
PEAK_ABLATION_LEVELS = ('L0', 'L4', 'L7')
PEAK_ABLATION_MULTIPLIERS = (0.0, 0.1, 1.0)
PEAK_ABLATION_READS = 250
PEAK_ABLATION_SEED = 3031

peak_ablation_rows = []
for level in PEAK_ABLATION_LEVELS:
    instance = instances[level]
    for encoding in ENCODINGS:
        base_weights = selected_weights[(level, encoding)]
        for p in PEAK_ABLATION_MULTIPLIERS:
            weights = dict(base_weights)
            weights['peak'] = p * instance['energy_scale'] / instance['peak_reference_mw'] ** 2
            run = anneal(instance, encoding, weights, num_reads=PEAK_ABLATION_READS,
                         seed=PEAK_ABLATION_SEED)
            schedule = run.pop('schedule')
            physical = physical_metrics(schedule, instance) if run['feasible'] else {}
            peak_ablation_rows.append({
                'level': level, 'encoding': encoding, 'peak_multiplier': p,
                'feasible': run['feasible'], 'feasible_fraction': run['feasible_fraction'],
                'energy_cost': run['energy_cost'], 'sampling_time_s': run['sampling_time_s'],
                'assignment_errors': run['assignment_errors'],
                'overloaded_slots': run['overloaded_slots'],
                'physical_total_cost': physical.get('physical_total_cost'),
                'physical_peak_grid_import_mw': physical.get('physical_peak_grid_import_mw'),
            })
peak_ablation = pd.DataFrame(peak_ablation_rows)
peak_ablation.to_csv(RESULTS_DIR / 'peak_ablation_paired.csv', index=False)
display(peak_ablation.round(4))

## 8. Results

### 8.1 QUBO formulation comparison

Structure of the two primary peak-off QUBOs. Slack bits grow with the number of time intervals and the bits needed to represent capacity; the unbalanced formulation has no slack variables. Report logical variables, couplers, coupling density and coefficient range. Structural counts alone do not establish annealing performance.

In [ ]:
structure_rows = []
for level in RUN_LEVELS:
    for encoding in ENCODINGS:
        qubo = build_qubo(instances[level], encoding, selected_weights[(level, encoding)])
        structure_rows.append({'level': level, 'encoding': encoding, **qubo_structure(qubo)})
structure = pd.DataFrame(structure_rows)
structure.to_csv(RESULTS_DIR / 'qubo_structure.csv', index=False)

wide = structure.pivot(index='level', columns='encoding',
                       values=['assignment_variables', 'slack_variables', 'logical_variables', 'quadratic_couplers'])
wide.columns = [f'{metric} ({encoding})' for metric, encoding in wide.columns]
wide['variable_reduction_pct'] = 100 * (1 - wide['logical_variables (unbalanced)'] / wide['logical_variables (slack)'])
wide['coupler_reduction_pct'] = 100 * (1 - wide['quadratic_couplers (unbalanced)'] / wide['quadratic_couplers (slack)'])
display(wide.round(1))
display(structure[['level', 'encoding', 'coupling_density', 'mean_degree', 'max_degree',
                   'max_abs_coefficient', 'coefficient_dynamic_range']].round(3))

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(13, 4))
positions = np.arange(len(RUN_LEVELS))
for offset, encoding, color in ((-0.2, 'slack', '#4c78a8'), (0.2, 'unbalanced', '#f58518')):
    subset = structure.set_index('level').loc[list(RUN_LEVELS)]
    subset = subset[subset['encoding'] == encoding]
    axes[0].bar(positions + offset, subset['logical_variables'], width=0.4, color=color, label=encoding)
    axes[1].bar(positions + offset, subset['quadratic_couplers'], width=0.4, color=color, label=encoding)
for axis, title in zip(axes, ('Logical variables', 'Quadratic couplers')):
    axis.set_xticks(positions, RUN_LEVELS)
    axis.set_yscale('log')
    axis.set_title(title)
    axis.grid(axis='y', alpha=0.3)
    axis.legend()
figure.suptitle('QUBO size: slack versus unbalanced formulation')
figure.tight_layout()
figure.savefig(FIGURES_DIR / 'qubo_size.png', dpi=150, bbox_inches='tight')
plt.show()

### 8.2 Solver comparison: feasibility, time and cost

- `feasible_fraction`: mean feasible-read fraction across three independent final annealing seeds; `feasible_fraction_sd` reports its across-seed standard deviation.
- `feasible_replicates`: number of final seeds that returned a feasible selected schedule.
- `sampling_time_s`: mean sampler-call time across final seeds; Gurobi time is its single solve time. These are implementation-specific runtimes, not a quantum-versus-classical speed comparison.
- `energy_cost`: median effective-price cost among feasible annealing replicates. The aligned MILP value is its incumbent cost.
- `energy_cost_gap_to_aligned_reference_pct`: descriptive difference from the aligned MILP incumbent, not an optimality gap for the unbalanced QUBO.
- `physical_total_cost`: median physical re-evaluation of feasible annealing schedules; `physical_cost_gap_to_full_milp_incumbent_pct` compares it with the full MILP incumbent.
- `mip_gap` and `optimality_proven` indicate whether a MILP result is an incumbent within tolerance or a proven optimum.
- Facility energy is identical across feasible schedules within an instance under fixed PUE and all-jobs-assigned; timing changes energy cost and the renewable/grid split.

In [ ]:
columns = ['level', 'method', 'feasible', 'feasible_fraction', 'feasible_fraction_sd',
           'feasible_replicates', 'n_replicates', 'sampling_time_s', 'tuning_time_s', 'energy_cost',
           'energy_cost_gap_to_aligned_reference_pct', 'physical_total_cost',
           'physical_cost_gap_to_full_milp_incumbent_pct', 'facility_energy_mwh',
           'grid_energy_mwh', 'renewable_energy_mwh', 'physical_peak_grid_import_mw',
           'assignment_errors', 'overloaded_slots', 'max_overload_gpu', 'mip_gap', 'optimality_proven']
display(final[[column for column in columns if column in final]].round(4))

In [ ]:
annealing = final[final['encoding'].isin(ENCODINGS)]
milp = final[final['encoding'].str.startswith('milp')]
colors = {'SA-slack': '#4c78a8', 'SA-unbalanced': '#f58518',
          'Gurobi QUBO-aligned MILP': '#54a24b', 'Gurobi full MILP': '#b279a2'}

figure, axes = plt.subplots(1, 3, figsize=(17, 4.2))
for method, group in final.groupby('method'):
    group = group.set_index('level').reindex(list(RUN_LEVELS))
    axes[0].plot(RUN_LEVELS, group['sampling_time_s'], marker='o', color=colors[method], label=method)
axes[0].set_yscale('log')
axes[0].set_title('Execution time (final run)')
axes[0].set_ylabel('seconds')

for method, group in annealing.groupby('method'):
    group = group.set_index('level').reindex(list(RUN_LEVELS))
    axes[1].plot(RUN_LEVELS, group['feasible_fraction'], marker='o', color=colors[method], label=method)
axes[1].set_ylim(-0.05, 1.05)
axes[1].set_title('Feasible fraction of annealing reads')

for method, group in annealing.groupby('method'):
    group = group.set_index('level').reindex(list(RUN_LEVELS))
    axes[2].plot(RUN_LEVELS, group['energy_cost_gap_to_aligned_reference_pct'], marker='o', color=colors[method], label=method)
axes[2].axhline(0, color='#54a24b', linestyle='--', linewidth=1, label='aligned MILP incumbent')
axes[2].set_title('Effective-energy cost difference vs aligned incumbent')
axes[2].set_ylabel('relative difference (%)')
for axis in axes:
    axis.grid(alpha=0.3)
    axis.legend(fontsize=8)
figure.tight_layout()
figure.savefig(FIGURES_DIR / 'solver_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### 8.3 Summary

The next cell prints concise per-level results. A method is marked as having found a feasible schedule if any of the three final seeds did so; the feasible-seed count and mean feasible-read fraction preserve the distinction. Do not report a cost for levels with no feasible selected schedule. For Gurobi, retain the MIP gap and `optimality_proven` flag.

In [ ]:
lines = []
for level in RUN_LEVELS:
    rows = final[final['level'] == level].set_index('encoding')
    parts = []
    for encoding in (*ENCODINGS, 'milp_aligned', 'milp_full'):
        if encoding not in rows.index:
            continue
        row = rows.loc[encoding]
        if not row['feasible']:
            errors, overloaded = row.get('assignment_errors'), row.get('overloaded_slots')
            detail = (f' ({int(errors)} assignment errors, {int(overloaded)} overloaded slots)'
                      if pd.notna(overloaded) else '')
            parts.append(f"{METHOD_LABELS[encoding]}: **infeasible**{detail}")
        elif encoding in ENCODINGS:
            parts.append(f"{METHOD_LABELS[encoding]}: {int(row['feasible_replicates'])}/{int(row['n_replicates'])} feasible seeds, {row['feasible_fraction']:.0%} mean feasible reads, "
                         f"gap {row['energy_cost_gap_to_aligned_reference_pct']:+.2f}%, {row['sampling_time_s']:.2f}s")
        else:
            parts.append(f"{METHOD_LABELS[encoding]}: {row['sampling_time_s']:.2f}s, gap {row['mip_gap'] or 0:.2%}, optimality proven={row['optimality_proven']}")
    sizes = structure[structure['level'] == level].set_index('encoding')['logical_variables']
    lines.append(f"- **{level}** ({instances[level]['name']}, {sizes['slack']} vs {sizes['unbalanced']} variables): "
                 + '; '.join(parts))
display(Markdown('\n'.join(lines)))